In [ ]:
import sqlite3 
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool 
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt, Command

llm = init_chat_model("openai:gpt-4o-mini")

conn = sqlite3.connect("memory2.db", check_same_thread=False)

config={
    "configurable":{
        "thread_id":"2",
    }
}

In [ ]:

class State(MessagesState):
    custum_stuff:str

graph_builder = StateGraph(State)

In [ ]:
@tool
def get_human_feedback(poem:str):
    """
    Asks the user form feedvack on the poem.
    Use this before returning the final response.
    """
    feedback = interrupt(f"Here is the poem, tell me what you think\n{poem}")
    return feedback

llm_with_tools = llm.bind_tools(
    tools=[
        get_human_feedback
    ]
)

def chatbot(state:State):
    response = llm.invoke(f"""
        You are an expert in making poems.
        Use the 'get_human_feedback' tool to get feedback on your poem.
        Only after you receive positive feedback you can return the final poem.
        ALWAYS ASK FOR FEEDBACK FIRST.
        Here is the conversation history:
        {state["messages"]}             
    """)
    return {
        "messages" : [response]
    }

In [ ]:
tool_node = ToolNode(
    tools=[
        get_human_feedback
    ]
)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools",tool_node)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot",tools_condition)
graph_builder.add_edge("tools","chatbot")


# 그래프 컴파일
# 그래프랑 edge들이 다~말이되는지화긘 (유효성검사)
graph = graph_builder.compile(
    checkpointer=SqliteSaver(conn),
)



In [ ]:
result = graph.invoke(
    {
        'messages': [{
            "role":"user", "content":"Please make a poem about Python code."
        }]
    },
    config=config
)

In [ ]:
for message in result["messages"]:
    message.pretty_print()

In [56]:
snapshot = graph.get_state(config)

snapshot.interrupts

()

In [ ]:
response = Command(
    resume="It looks good!"
)

result = graph.invoke(
    response,
    config=config
)

for message in result["messages"]:
    message.pretty_print()